# Windowed VMD + Gaussian-Kernel (GK) Mode Selection — Generic Template

**GENERIC MODEL — READ BEFORE USE.** This notebook is a sex- and window-length-agnostic
template for the windowed-VMD + Gaussian-Kernel mode-selection pipeline described in the
paper's Methods section. It must be adapted to one specific sex (`FEMALE` or `MALE`) and one
specific window length (`500ms`, `1s`, or `2s`) before running: edit the `SEX`, `WINDOW_LABEL`,
`features_dir`, and `output_dir` values in the configuration block at the top of Step 1 to match
the combination you are running, then run all cells top to bottom. Each sex x window-length
combination used in the paper is a separate run of this same template, not a separate notebook.

**Pipeline overview (per-window then subject-level):**

```
Raw audio (DAICWOZ_split)
        │
        ▼
[Segmentation]  WINDOW_LENGTH_S-second non-overlapping windows
        │  features already extracted upstream (windowed VMD, K=8)
        ▼
[Windowed VMD, K=8]  per window: energy(8,) | frequency(8,) | stability(8,)
              concatenated → feature matrix (n_windows, 24) per subject
        │
        ▼
[Step 1]  Load raw feature matrices — separate TRAIN/TEST, DEP/HEALTH
        │
        ▼
[Step 2]  Gaussian Kernel (GK) mode selection
          • Build mode-vector R³ = [energy_k, freq_k, stab_k] per window
          • Compute RBF cross-kernel score between DEP and HEALTH clouds (TRAIN only)
          • Low score → well-separated → discriminative mode
          • Select N_SELECT modes with lowest score out of K=8 (4 for FEMALE, 5 for MALE)
        │
        ▼
[Step 3]  Global normalization + subject-level aggregation
          • Fit StandardScaler on ALL concatenated TRAIN windows (no per-subject fit)
          • For each subject: transform windows → apply selected modes → (n_windows, 3*N_SELECT)
          • Compute 6 statistics on X_sel and Δ(X_sel):
            skewness | kurtosis | std | mean | median | IQR
          • Result: one vector of 6 * 2 * 3*N_SELECT features per subject
        │
        ▼
[Step 4]  Subject-level classification — GridSearchCV (balanced_accuracy)
          • 8 models: DecisionTree, RandomForest, GradientBoosting, SVM, XGBoost,
            LogisticRegression, KNN, NaiveBayes
          • StratifiedKFold(5) on TRAIN subjects
          • Evaluate on TEST subjects (no window-level leakage)
        │
        ▼
[Step 5]  Stacking ensembles (2- and 3-base-model combinations)
        │
        ▼
[Step 6]  Youden's J threshold calibration (out-of-fold TRAIN probabilities)
[Step 6B/6C]  Leakage-free (TRAIN-CV) architecture selection, redirected into Step 7
        │
        ▼
[Step 7]  Winner's-curse / model-selection-bias check (repeated-split, out-of-fold)
        │
        ▼
[Step 8]  Save results to CSV
```

**Key design choices:**
- Classification unit: **subject** (not window) — one prediction per participant
- GK selection uses TRAIN data only — no leakage
- Global scaler fitted on TRAIN windows only — no leakage
- Non-overlapping windows — independent samples within each subject
- Discontinuity-aware chunking: windowing and order-dependent features (stability, Δ) never
  cross a chunk boundary (Ellie turn, long pause, or manual silence cut). Step 1 loads each
  subject's `is_chunk_start` boolean array; Step 3's `compute_delta` resets to 0 at every
  chunk start, not only at the start of each subject's recording.

## Step 1 — Load raw windowed-VMD features

Load per-subject `.npy` arrays for `energy`, `frequency`, and `stability` (each `(n_windows, K)`,
K=8) and concatenate horizontally to obtain `(n_windows, 24)` per subject. Also loads each
subject's `is_chunk_start` boolean array (same length as `n_windows`), used by Step 3's
`compute_delta`.

Subjects are split into four lists: TRAIN/TEST × DEP/HEALTH. No data from TEST subjects is used
in Steps 2–3 (no leakage).

In [ ]:
# =====================================================================
# Step 1 -- Load per-subject windowed-VMD features (energy/frequency/
# stability) plus each subject's is_chunk_start array.
# =====================================================================
# CONFIGURATION -- EDIT THESE VALUES FOR EACH SEX x WINDOW-LENGTH RUN OF
# THIS TEMPLATE. Everything below this block is generic.
# =====================================================================

from pathlib import Path
import numpy as np
import pandas as pd

SEX          = "FEMALE"      # "FEMALE" or "MALE"
WINDOW_LABEL = "1s"          # "500ms", "1s", or "2s"

STVMD_K  = 8                                 # fixed across all sex / window-length combinations (Section 2.4)
N_SELECT = {"FEMALE": 4, "MALE": 5}[SEX]     # GK-selected mode count (Section 2.6)
WINDOW_LENGTH_S = {"500ms": 0.5, "1s": 1.0, "2s": 2.0}[WINDOW_LABEL]

# Local paths to the windowed-VMD feature directories -- EDIT to match your setup
features_dir = Path(rf"D:\DAICWOZ_stvmd_{WINDOW_LABEL}_k{STVMD_K}")
output_dir   = Path(rf"D:\DAICWOZ_stvmd_{WINDOW_LABEL}_k{STVMD_K}_gk_global")

manifest_path = features_dir / "processed_manifest.csv"
features      = ["energy", "frequency", "stability"]
GRUPOS_TARGET = [f"{SEX}_DEP", f"{SEX}_HEALTH"]
dep_label, health_label = GRUPOS_TARGET

manifest = pd.read_csv(str(manifest_path))
manifest = manifest[manifest['grupo'].isin(GRUPOS_TARGET)].copy()

train_dep_raws,    train_dep_ids,    train_dep_ischunk    = [], [], []
train_health_raws, train_health_ids, train_health_ischunk = [], [], []
test_dep_raws,     test_dep_ids,     test_dep_ischunk     = [], [], []
test_health_raws,  test_health_ids,  test_health_ischunk  = [], [], []

for _, row in manifest.iterrows():
    subject_id = str(row['subject_id'])
    split      = row['split']
    grupo      = row['grupo']
    src_dir    = features_dir / f"{split}_{grupo}"
    arrays, missing = [], False
    for feat in features:
        path = src_dir / f"{subject_id}_{feat}_k{STVMD_K}.npy"
        if not path.exists():
            print(f'Not found: {path.name}')
            missing = True; break
        arrays.append(np.load(str(path)))

    ischunk_path = src_dir / f"{subject_id}_is_chunk_start_k{STVMD_K}.npy"
    if not ischunk_path.exists():
        print(f'Not found: {ischunk_path.name}')
        missing = True

    if missing:
        continue

    X_raw = np.concatenate(arrays, axis=1)   # (n_windows, 24)
    is_chunk_start = np.load(str(ischunk_path))
    assert is_chunk_start.shape[0] == X_raw.shape[0], (
        f"{subject_id}: X_raw ({X_raw.shape[0]}) / is_chunk_start "
        f"({is_chunk_start.shape[0]}) length mismatch"
    )

    if   split == "TRAIN" and grupo == dep_label:
        train_dep_raws.append(X_raw);    train_dep_ids.append(subject_id)
        train_dep_ischunk.append(is_chunk_start)
    elif split == "TRAIN" and grupo == health_label:
        train_health_raws.append(X_raw); train_health_ids.append(subject_id)
        train_health_ischunk.append(is_chunk_start)
    elif split == "TEST"  and grupo == dep_label:
        test_dep_raws.append(X_raw);     test_dep_ids.append(subject_id)
        test_dep_ischunk.append(is_chunk_start)
    elif split == "TEST"  and grupo == health_label:
        test_health_raws.append(X_raw);  test_health_ids.append(subject_id)
        test_health_ischunk.append(is_chunk_start)

print(f'TRAIN DEP    : {len(train_dep_raws)} subjects')
print(f'TRAIN HEALTH : {len(train_health_raws)} subjects')
print(f'TEST DEP     : {len(test_dep_raws)} subjects')
print(f'TEST HEALTH  : {len(test_health_raws)} subjects')
if train_dep_raws:
    print(f'Example shape (first DEP TRAIN subject): {train_dep_raws[0].shape}  -> (n_windows, 24)')

## Step 1.5 — Basic descriptive statistics (TRAIN/TEST × DEP/HEALTH)

Before any GK selection or feature extraction, build a per-subject table (one row per subject
with its window count / duration) and a group-level summary (n subjects, total windows,
mean/std/min/median/max windows per subject). Duration is derived from `n_windows *
WINDOW_LENGTH_S` (windows are non-overlapping, so this holds for any window length).

In [ ]:
groups = {
    ("TRAIN", dep_label):    (train_dep_raws,    train_dep_ids),
    ("TRAIN", health_label): (train_health_raws, train_health_ids),
    ("TEST",  dep_label):    (test_dep_raws,     test_dep_ids),
    ("TEST",  health_label): (test_health_raws,  test_health_ids),
}

subject_rows = []
for (split, grupo), (X_list, ids_list) in groups.items():
    for X, sid in zip(X_list, ids_list):
        subject_rows.append({
            "subject_id":  sid,
            "split":       split,
            "grupo":       grupo,
            "n_windows":   X.shape[0],
            "duration_s":  X.shape[0] * WINDOW_LENGTH_S,
        })

subjects_df = pd.DataFrame(subject_rows)
print(f"=== Per-subject statistics ({WINDOW_LENGTH_S}s per window, non-overlapping) ===")
print(subjects_df.to_string(index=False))

summary_df = subjects_df.groupby(["split", "grupo"])["n_windows"].agg(
    n_subjects="count", total_windows="sum", mean_windows="mean",
    std_windows="std", min_windows="min", median_windows="median", max_windows="max"
).round(1).reset_index()

print("\n=== Summary by group ===")
print(summary_df.to_string(index=False))

print("\n=== Class balance ===")
for split in ["TRAIN", "TEST"]:
    n_dep    = summary_df.loc[(summary_df.split == split) & (summary_df.grupo == dep_label),    "n_subjects"].iloc[0]
    n_health = summary_df.loc[(summary_df.split == split) & (summary_df.grupo == health_label), "n_subjects"].iloc[0]
    print(f"  {split}: {int(n_dep)} dep / {int(n_health)} health  "
          f"(ratio dep:health = 1:{n_health/n_dep:.2f})")

## Step 2 — Gaussian Kernel (GK) mode selection

For each of the K=8 windowed-VMD modes, build a **mode-vector** in R³ = `[energy_k, freq_k,
stab_k]` for every window, then concatenate all TRAIN windows per class.

Class separability is measured via the **RBF cross-kernel**:

$$K[i,j] = \exp\!\left(-\frac{\|A_i - B_j\|^2}{2\sigma^2}\right)$$

where σ² is estimated with the **median heuristic** on a random subsample (no TEST data used).

- **High score** → overlapping distributions → non-discriminative mode
- **Low score** → separated distributions → discriminative mode

The `N_SELECT` modes with the lowest score are selected for feature extraction (`N_SELECT=4`
for FEMALE, `N_SELECT=5` for MALE, per Section 2.6).

In [ ]:
def mode_vector(X_list, k):
    return np.concatenate(
        [np.stack([X[:, k], X[:, k + STVMD_K], X[:, k + 2 * STVMD_K]], axis=1)
         for X in X_list], axis=0).astype(np.float64)

def rbf_cross_score(A, B, chunk_size=200):
    rng    = np.random.default_rng(42)
    n_samp = min(2000, len(A), len(B))
    Asamp  = A[rng.choice(len(A), n_samp, replace=False)]
    Bsamp  = B[rng.choice(len(B), n_samp, replace=False)]
    diff2  = ((Asamp[:, None, :] - Bsamp[None, :, :]) ** 2).sum(axis=2)
    sigma2 = float(np.median(diff2)) + 1e-12
    # Chunked kernel to avoid OOM with large window counts
    k_sum = 0.0
    for i in range(0, len(A), chunk_size):
        A_chunk = A[i:i + chunk_size]
        d2 = ((A_chunk[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)
        k_sum += np.exp(-d2 / (2 * sigma2)).sum()
    return k_sum / (len(A) * len(B)), sigma2

scores = []
for k in range(STVMD_K):
    dep_k         = mode_vector(train_dep_raws,    k)
    health_k      = mode_vector(train_health_raws, k)
    score, sigma2 = rbf_cross_score(dep_k, health_k)
    scores.append(score)
    print(f'  Mode {k}: score={score:.4f}  sigma2={sigma2:.2f}'
          f'  (dep={len(dep_k)} win, health={len(health_k)} win)')

scores          = np.array(scores)
ranking         = np.argsort(scores)
selected_modes  = sorted(ranking[:N_SELECT].tolist())
discarded_modes = sorted(ranking[N_SELECT:].tolist())

print(f'\nSelected modes  (lowest score, most discriminative): {selected_modes}')
print(f'Discarded modes (highest score, most overlapping)   : {discarded_modes}')
print(f'\nScores per mode: {np.round(scores, 4).tolist()}')

## Step 3 — Global normalization and subject-level feature extraction

**Normalization strategy:** `StandardScaler` is fitted on **all TRAIN windows concatenated** (not
per-subject). This avoids inflating intra-subject variance estimates and ensures a single
consistent scale across all subjects.

**Feature extraction per subject:**
1. Transform windows with the global scaler → `(n_windows, 24)`
2. Select columns for the `N_SELECT` GK-chosen modes → `(n_windows, 3*N_SELECT)`
3. Compute the discrete temporal derivative Δ: `delta[t] = X[t] - X[t-1]`, reset to 0 at every
   chunk start (`is_chunk_start`)
4. Compute statistics over the temporal axis (axis=0):

| Statistic     | On X_sel | On delta | Rationale |
|---------------|:---:|:---:|---|
| Skewness      | ✓   | ✓   | Tail asymmetry |
| Kurtosis      | ✓   | ✓   | Tail weight vs centre |
| Std           | ✓   | ✓   | Overall dispersion |
| Mean          | ✓   | ✓   | Central tendency |
| Median        | ✓   | ✓   | Robust central tendency |
| IQR (p75−p25) | ✓   | ✓   | Robust dispersion |

**Output:** one vector of `6 * 2 * 3*N_SELECT` features per subject (144 for FEMALE's
N_SELECT=4, 180 for MALE's N_SELECT=5).

Note: an earlier version of this step also computed a 7th statistic (a 10-bin histogram mode).
Ablation testing showed it added no genuine discriminative signal, so it is not computed here —
this matches the 6-statistic design reported in Section 2.7 of the paper.

In [ ]:
from scipy import stats as sp_stats
from sklearn.preprocessing import StandardScaler

output_dir.mkdir(parents=True, exist_ok=True)

labels_map = {dep_label: 1, health_label: 0}


def compute_delta(arr, is_chunk_start=None):
    delta     = np.zeros_like(arr)
    delta[1:] = arr[1:] - arr[:-1]
    if is_chunk_start is not None:
        delta[is_chunk_start] = 0   # discontinuity reset (window 0 is
                                     # always flagged too, so this
                                     # subsumes the "first window" rule)
    return delta


def subject_features_gk(X_norm, sel_modes, is_chunk_start):
    """
    Returns : (6 * 2 * 3*len(sel_modes),)  [6 stats x 2 (X_sel + delta) x 3*len(sel_modes) cols]
    """
    cols  = [c for k in sel_modes for c in (k, k + STVMD_K, k + 2 * STVMD_K)]
    X_sel = X_norm[:, cols]          # (n_windows, 3*len(sel_modes))
    delta = compute_delta(X_sel, is_chunk_start=is_chunk_start)
    return np.concatenate([
        sp_stats.skew(X_sel, axis=0),
        sp_stats.kurtosis(X_sel, axis=0),
        X_sel.std(axis=0),
        X_sel.mean(axis=0),
        np.median(X_sel, axis=0),
        np.percentile(X_sel, 75, axis=0) - np.percentile(X_sel, 25, axis=0),
        sp_stats.skew(delta, axis=0),
        sp_stats.kurtosis(delta, axis=0),
        delta.std(axis=0),
        delta.mean(axis=0),
        np.median(delta, axis=0),
        np.percentile(delta, 75, axis=0) - np.percentile(delta, 25, axis=0),
    ])


# Fit global scaler on ALL concatenated TRAIN windows
X_all_train   = np.vstack(train_dep_raws + train_health_raws)
scaler_global = StandardScaler()
scaler_global.fit(X_all_train)
print(f'Global scaler fitted on {X_all_train.shape[0]} TRAIN windows')


def extract_group(X_raws, ids, ischunk_list, grupo):
    X_list, y_list, id_list = [], [], []
    for X_raw, sid, is_chunk_start in zip(X_raws, ids, ischunk_list):
        X_norm = scaler_global.transform(X_raw)
        X_list.append(subject_features_gk(X_norm, selected_modes, is_chunk_start))
        y_list.append(labels_map[grupo])
        id_list.append(sid)
        print(f'  {grupo:15s} | {sid:>4s}: {X_raw.shape[0]:>4} windows -> ({len(X_list[-1])},)')
    return X_list, y_list, id_list


print('\n[TRAIN]')
Xtrd, ytrd, idtrd = extract_group(train_dep_raws,    train_dep_ids,    train_dep_ischunk,    dep_label)
Xtrh, ytrh, idtrh = extract_group(train_health_raws, train_health_ids, train_health_ischunk, health_label)
print('\n[TEST]')
Xted, yted, idted = extract_group(test_dep_raws,     test_dep_ids,     test_dep_ischunk,     dep_label)
Xteh, yteh, idteh = extract_group(test_health_raws,  test_health_ids,  test_health_ischunk,  health_label)

X_train   = np.vstack(Xtrd + Xtrh);  y_train   = np.array(ytrd + ytrh)
ids_train = np.array(idtrd + idtrh)
X_test    = np.vstack(Xted + Xteh);  y_test    = np.array(yted + yteh)
ids_test  = np.array(idted + idteh)

## Step 4 — Subject-level classification with GridSearchCV

Eight classifiers are tuned with `GridSearchCV` using **balanced accuracy** as the scoring metric
to compensate for the class imbalance between DEP and HEALTH subjects in TRAIN.

**Cross-validation:** `StratifiedKFold(5)` where each sample is a **subject** — no window-level
leakage is possible since windows from the same subject are never split.

**Class imbalance handling:**
- DecisionTree, RandomForest, SVM, LogisticRegression: `class_weight='balanced'`
- GradientBoosting: `sample_weight` balanced (via `compute_sample_weight`)
- XGBoost: `scale_pos_weight = n_health / n_dep`
- KNN, NaiveBayes (GaussianNB): no native class weighting in scikit-learn — left unweighted
  (KNN's `weights='distance'` option is tuned as a partial mitigation)

**Confidence intervals:** 95% CI via **percentile bootstrap** (n=10000 resamplings of the TEST
subjects). Metrics reported: Precision, Recall, Specificity, F1, ROC-AUC.

**Plots:** ROC curve and Precision-Recall curve for all models on the test set.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE   = 42
n_dep_train    = int((y_train == 1).sum())
n_health_train = int((y_train == 0).sum())
N_SPLITS       = min(5, n_dep_train, n_health_train)
cv             = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f'X_train : {X_train.shape}  -- {n_dep_train} dep, {n_health_train} health')
print(f'X_test  : {X_test.shape}   -- {(y_test==1).sum()} dep, {(y_test==0).sum()} health')
print(f'CV: StratifiedKFold(n_splits={N_SPLITS}) — each sample is a subject, no leakage')

fit_params_per_model = {
    'GradientBoosting': {'sample_weight': compute_sample_weight('balanced', y_train)}
}

def compute_metrics(model, X_t, y_t):
    y_pred  = model.predict(X_t)
    y_proba = model.predict_proba(X_t)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_t, y_pred).ravel()
    return {
        'y_pred': y_pred, 'y_proba': y_proba,
        'accuracy':     accuracy_score(y_t, y_pred),
        'bal_accuracy': balanced_accuracy_score(y_t, y_pred),
        'precision':    precision_score(y_t, y_pred, zero_division=0),
        'recall':       recall_score(y_t, y_pred, zero_division=0),
        'specificity':  tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'f1':           f1_score(y_t, y_pred, zero_division=0),
        'roc_auc':      roc_auc_score(y_t, y_proba),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

def bootstrap_ci(y_true, y_pred, y_proba, n_boot=10000, alpha=0.05, seed=0):
    """95% CI via percentile bootstrap on the test set subjects."""
    rng  = np.random.default_rng(seed)
    n    = len(y_true)
    boot = {k: [] for k in ['accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc']}
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt, yp, ypr = y_true[idx], y_pred[idx], y_proba[idx]
        if len(np.unique(yt)) < 2:
            continue
        tn_b, fp_b, fn_b, tp_b = confusion_matrix(yt, yp).ravel()
        boot['accuracy'].append(accuracy_score(yt, yp))
        boot['precision'].append(precision_score(yt, yp, zero_division=0))
        boot['recall'].append(recall_score(yt, yp, zero_division=0))
        boot['specificity'].append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0.0)
        boot['f1'].append(f1_score(yt, yp, zero_division=0))
        boot['roc_auc'].append(roc_auc_score(yt, ypr))
    return {k: (np.percentile(v, 100*alpha/2), np.percentile(v, 100*(1-alpha/2)))
            for k, v in boot.items()}

models_grids = {
    'DecisionTree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        {'criterion': ['gini', 'entropy'],
         'max_depth': [3, 5, 10, 15, None],
         'min_samples_split': [2, 5, 10, 20],
         'min_samples_leaf': [1, 2, 4]},
    ),
    'RandomForest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
        {'n_estimators': [100, 200, 500],
         'max_depth': [5, 10, 20, None],
         'max_features': ['sqrt', 'log2'],
         'min_samples_split': [2, 5, 10],
         'min_samples_leaf': [1, 2, 4]},
    ),
    'GradientBoosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {'n_estimators': [100, 200, 300],
         'learning_rate': [0.05, 0.1, 0.2],
         'max_depth': [2, 3, 5],
         'subsample': [0.7, 0.85, 1.0],
         'min_samples_leaf': [1, 2, 4]},
    ),
    'SVM': (
        SVC(random_state=RANDOM_STATE, probability=True, class_weight='balanced'),
        {'C': [0.01, 0.1, 1, 10, 100],
         'kernel': ['rbf', 'linear'],
         'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]},
    ),
    'XGBoost': (
        xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_jobs=-1,
                          scale_pos_weight=n_health_train / n_dep_train),
        {'n_estimators': [100, 200, 300],
         'learning_rate': [0.05, 0.1, 0.2],
         'max_depth': [2, 3, 5],
         'subsample': [0.7, 0.85, 1.0],
         'colsample_bytree': [0.7, 1.0],
         'min_child_weight': [1, 3, 5],
         'reg_alpha': [0, 0.1, 1.0]},
    ),
    'LogisticRegression': (
        LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=2000),
        {'C': [0.001, 0.01, 0.1, 1, 10, 100],
         'penalty': ['l1', 'l2'],
         'solver': ['liblinear']},
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'n_neighbors': [3, 5, 7, 9, 11, 15],
         'weights': ['uniform', 'distance'],
         'metric': ['euclidean', 'manhattan']},
    ),
    'NaiveBayes': (
        GaussianNB(),
        {'var_smoothing': np.logspace(0, -9, 10)},
    ),
}

results = {}
for model_name, (estimator, param_grid) in models_grids.items():
    print(f"\n{'='*60}\n  {model_name}\n{'='*60}")
    gs = GridSearchCV(estimator, param_grid, cv=cv, scoring='balanced_accuracy',
                      n_jobs=-1, verbose=0, refit=True)
    fit_kw = fit_params_per_model.get(model_name, {})
    gs.fit(X_train, y_train, **fit_kw)
    m  = compute_metrics(gs.best_estimator_, X_test, y_test)
    ci = bootstrap_ci(y_test, m['y_pred'], m['y_proba'])
    detail = pd.DataFrame({
        'subject': ids_test, 'true': y_test, 'pred': m['y_pred'],
        'proba': m['y_proba'].round(3), 'correct': y_test == m['y_pred'],
    })
    results[model_name] = {
        'best_params': gs.best_params_, 'cv_score': gs.best_score_,
        'y_true': y_test, 'y_proba': m['y_proba'], 'ci': ci,
        **{k: v for k, v in m.items() if k not in ('y_pred','y_proba','tn','fp','fn','tp')},
    }
    def fmt(k): return f"{m[k]:.3f} [{ci[k][0]:.3f}–{ci[k][1]:.3f}]"
    print(f'\n  Best parameters : {gs.best_params_}')
    print(f'\n  Confusion matrix:')
    print(f'                 Pred Health  Pred Dep')
    print(f"  True Health        {m['tn']:>3}         {m['fp']:>3}")
    print(f"  True Dep           {m['fn']:>3}         {m['tp']:>3}")
    print(f'\n  Test metrics  (value [95% CI bootstrap]):')
    print(f"    Accuracy     : {fmt('accuracy')}")
    print(f"    Precision    : {fmt('precision')}")
    print(f"    Recall/Sens. : {fmt('recall')}")
    print(f"    Specificity  : {fmt('specificity')}")
    print(f"    F1           : {fmt('f1')}")
    print(f"    ROC AUC      : {fmt('roc_auc')}")
    print(f'\n  Per-subject detail:')
    print(detail.to_string(index=False))

print('\n\n=== FINAL SUMMARY  (sorted by ROC AUC — value [95% CI bootstrap]) ===')
hdr = (f"{'Model':<20} {'ROC':>22} {'Accuracy':>22} {'Recall':>22} "
       f"{'Prec':>22} {'Spec':>22} {'F1':>22}")
print(hdr)
print('-' * len(hdr))
for name, res in sorted(results.items(), key=lambda x: x[1]['roc_auc'], reverse=True):
    ci = res['ci']
    def f(k): return f"{res[k]:.3f} [{ci[k][0]:.3f}–{ci[k][1]:.3f}]"
    print(f"{name:<20} {f('roc_auc'):>22} {f('accuracy'):>22} {f('recall'):>22} "
          f"{f('precision'):>22} {f('specificity'):>22} {f('f1'):>22}")

### Export ROC data

Saves each model's TEST-set true labels, predicted probabilities, and ROC-AUC to a JSON file
(`roc_export_<sex>_<window>.json`), used later to build the paper's ROC comparison figure.

In [ ]:
import json

roc_export = {}
for name, res in results.items():
    roc_export[name] = {
        'y_true':  [int(v) for v in res['y_true']],
        'y_proba': [float(v) for v in res['y_proba']],
        'roc_auc': float(res['roc_auc']),
    }

roc_export_path = f'roc_export_{SEX.lower()}_{WINDOW_LABEL}.json'
with open(roc_export_path, 'w') as f:
    json.dump(roc_export, f, indent=2)

print(f'Saved {roc_export_path} with', len(roc_export), 'models')
for name, d in roc_export.items():
    print(f"  {name}: {len(d['y_true'])} subjects, AUC={d['roc_auc']:.4f}")

## Step 4B — Nested cross-validation sanity check

An additional leakage check: GK mode selection, the global scaler, and hyperparameter tuning
are all refit **inside** each outer fold (5-fold `StratifiedKFold`) using only that fold's
TRAIN subjects — nothing from Steps 2/3/4 is computed once on all of TRAIN and reused across
folds. Reports the honest, nested AUC-ROC (mean ± SD across outer folds) and checks how stable
the GK mode ranking and the winning model are across folds.

In [ ]:
# =====================================================================
# Step 4B -- Nested cross-validation sanity check (no preprocessing leakage)
# =====================================================================
# DISCONTINUITY-AWARE UPDATE: subject_features_gk / compute_delta now
# require each subject's is_chunk_start array (so Delta-X resets at
# genuine discontinuities, not just at subject start -- same rule as
# Step 3). This cell threads train_dep_ischunk / train_health_ischunk
# (Step 1) through every fold, the same way it already threads raws/ids/y.
# =====================================================================

from sklearn.model_selection import StratifiedKFold as _OuterKFold
from sklearn.base import clone
import pandas as pd

N_OUTER = 5
outer_cv = _OuterKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)

# All TRAIN subjects together, so the outer split can stratify by label.
all_train_raws    = train_dep_raws + train_health_raws
all_train_ids     = train_dep_ids + train_health_ids
all_train_ischunk = train_dep_ischunk + train_health_ischunk
all_train_y       = np.array([1] * len(train_dep_raws) + [0] * len(train_health_raws))


def _fold_mode_ranking(raws_fit, y_fit):
    """Recompute the GK mode ranking using ONLY this fold's own fit subjects (mirrors Step 2, but scoped to raws_fit/y_fit)."""
    dep_raws    = [r for r, y in zip(raws_fit, y_fit) if y == 1]
    health_raws = [r for r, y in zip(raws_fit, y_fit) if y == 0]
    scores = []
    for k in range(STVMD_K):
        dep_k    = mode_vector(dep_raws, k)
        health_k = mode_vector(health_raws, k)
        score, _ = rbf_cross_score(dep_k, health_k)
        scores.append(score)
    scores  = np.array(scores)
    ranking = np.argsort(scores)
    return sorted(ranking[:N_SELECT].tolist())


def _fold_feature_matrix(raws, ids, y, ischunk, sel_modes, scaler):
    """Build the (n_subjects, n_features) feature matrix for a list of subjects, using a fold-specific mode selection / scaler (mirrors Step 3's subject_features_gk, parameterized per fold). `ischunk` is each subject's is_chunk_start array, same order as `raws`."""
    X_list = []
    for X_raw, is_chunk_start in zip(raws, ischunk):
        X_norm = scaler.transform(X_raw)
        X_list.append(subject_features_gk(X_norm, sel_modes, is_chunk_start))
    return np.vstack(X_list), np.array(y), np.array(ids)


outer_results = []   # one row per (outer fold, model)

for fold_i, (fit_idx, val_idx) in enumerate(outer_cv.split(all_train_raws, all_train_y)):
    raws_fit    = [all_train_raws[i]    for i in fit_idx]
    ids_fit     = [all_train_ids[i]     for i in fit_idx]
    ischunk_fit = [all_train_ischunk[i] for i in fit_idx]
    y_fit       = [int(all_train_y[i])  for i in fit_idx]
    raws_val    = [all_train_raws[i]    for i in val_idx]
    ids_val     = [all_train_ids[i]     for i in val_idx]
    ischunk_val = [all_train_ischunk[i] for i in val_idx]
    y_val       = [int(all_train_y[i])  for i in val_idx]

    print(f'\n{"="*70}\nOUTER FOLD {fold_i + 1}/{N_OUTER}  '
          f'(fit: {len(fit_idx)} subjects, val: {len(val_idx)} subjects)\n{"="*70}')

    # --- Mode ranking, refit on this fold's fit subjects only ---
    sel_modes_fold = _fold_mode_ranking(raws_fit, y_fit)
    print(f'  Selected modes (fold-local): {sel_modes_fold}')

    # --- Scaler, refit on this fold's fit windows only ---
    X_all_fit   = np.vstack(raws_fit)
    scaler_fold = StandardScaler().fit(X_all_fit)

    # --- Build fold-local feature matrices ---
    X_fit, y_fit_arr, _ = _fold_feature_matrix(
        raws_fit, ids_fit, y_fit, ischunk_fit, sel_modes_fold, scaler_fold)
    X_val, y_val_arr, ids_val_arr = _fold_feature_matrix(
        raws_val, ids_val, y_val, ischunk_val, sel_modes_fold, scaler_fold)

    # --- Inner CV for hyperparameter tuning, on this fold's fit subjects only ---
    n_dep_fit    = int((y_fit_arr == 1).sum())
    n_health_fit = int((y_fit_arr == 0).sum())
    inner_splits = max(2, min(5, n_dep_fit, n_health_fit))
    inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=RANDOM_STATE)
    fit_params_fold = {
        'GradientBoosting': {'sample_weight': compute_sample_weight('balanced', y_fit_arr)}
    }

    for model_name, (estimator, param_grid) in models_grids.items():
        gs = GridSearchCV(clone(estimator), param_grid, cv=inner_cv,
                           scoring='balanced_accuracy', n_jobs=-1, refit=True)
        fit_kw = fit_params_fold.get(model_name, {})
        gs.fit(X_fit, y_fit_arr, **fit_kw)
        m = compute_metrics(gs.best_estimator_, X_val, y_val_arr)
        outer_results.append({
            'fold': fold_i, 'model': model_name,
            'n_fit': len(fit_idx), 'n_val': len(val_idx),
            'selected_modes': tuple(sel_modes_fold),
            'roc_auc': m['roc_auc'], 'bal_accuracy': m['bal_accuracy'],
            'f1': m['f1'], 'best_params': gs.best_params_,
        })
        print(f'    {model_name:18s}  ROC-AUC={m["roc_auc"]:.3f}  '
              f'BalAcc={m["bal_accuracy"]:.3f}  F1={m["f1"]:.3f}')

outer_df = pd.DataFrame(outer_results)


# =====================================================================
# Step 4B (continued) -- Aggregation and comparison to the paper's
# already-reported, non-nested estimates
# =====================================================================
best_idx_per_fold = outer_df.groupby('fold')['roc_auc'].idxmax()
fold_winners = outer_df.loc[best_idx_per_fold].reset_index(drop=True)

print(f'\n{"="*70}\nPer-fold winning model (nested, leakage-free)\n{"="*70}')
print(fold_winners[['fold', 'model', 'n_val', 'selected_modes',
                     'roc_auc', 'bal_accuracy', 'f1']].to_string(index=False))

mean_auc = fold_winners.roc_auc.mean()
sd_auc   = fold_winners.roc_auc.std(ddof=1)
print(f'\nNested-CV honest AUC-ROC across {N_OUTER} outer folds '
      f'(winning model per fold):')
print(f'  mean = {mean_auc:.3f}   sd = {sd_auc:.3f}   '
      f'range = [{fold_winners.roc_auc.min():.3f}, {fold_winners.roc_auc.max():.3f}]')

print(f'\nMode-ranking stability across outer folds:')
for row in fold_winners.itertuples():
    print(f'  fold {row.fold}: modes {row.selected_modes}')

print(f'\nWinning-model stability across outer folds:')
print(fold_winners['model'].value_counts().to_string())

### ROC and Precision-Recall curves

Plots each Step 4 model's ROC curve and Precision-Recall curve on the TEST set, for visual
inspection alongside the numeric summary above.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

COLORS = {
    'DecisionTree':       '#1f77b4',
    'RandomForest':       '#2ca02c',
    'GradientBoosting':   '#ff7f0e',
    'SVM':                '#d62728',
    'XGBoost':            '#9467bd',
    'LogisticRegression': '#8c564b',
    'KNN':                '#e377c2',
    'NaiveBayes':         '#17becf',
}

model_order = sorted(results.keys(), key=lambda x: results[x]['roc_auc'], reverse=True)
n_models    = len(model_order)
prevalence  = y_test.mean()

fig, axes = plt.subplots(n_models, 2, figsize=(12, 3.5 * n_models))
fig.suptitle('ROC and Precision-Recall curves per model — Test set', fontsize=13, y=1.01)

for i, name in enumerate(model_order):
    res   = results[name]
    color = COLORS[name]
    y_true, y_proba = res['y_true'], res['y_proba']

    # --- ROC ---
    ax = axes[i, 0]
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'AUC = {res["roc_auc"]:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('FPR (1 − Specificity)', fontsize=9)
    ax.set_ylabel('TPR (Sensitivity)', fontsize=9)
    ax.set_title(f'{name} — ROC', fontsize=10)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)

    # --- Precision-Recall ---
    ax = axes[i, 1]
    prec, rec, _ = precision_recall_curve(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)
    ax.plot(rec, prec, color=color, lw=2,
            label=f'AP = {ap:.3f}')
    ax.axhline(prevalence, color='k', linestyle='--', lw=1,
               label=f'Baseline (prev={prevalence:.2f})')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('Recall (Sensitivity)', fontsize=9)
    ax.set_ylabel('Precision (PPV)', fontsize=9)
    ax.set_title(f'{name} — Precision-Recall', fontsize=10)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5 — Stacking ensembles

Two families of stacking ensembles, built from the models tuned in Step 4 (reusing each one's
`best_params`):
- **2-base-model stacking**: a curated list of `(base_1, base_2) -> meta` combinations.
- **3-base-model stacking**: every 3-model combination drawn from
  `{DecisionTree, RandomForest, GradientBoosting, XGBoost, NaiveBayes, LogisticRegression}`,
  each combined with a `GradientBoosting` or `XGBoost` meta-model.

Both use `StackingClassifier(stack_method='predict_proba')`, evaluated on TEST the same way as
Step 4.

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.naive_bayes import GaussianNB

def make_estimator(name):
    """Build a fresh estimator for `name` reusing its best_params from Step 4 GridSearchCV."""
    bp = results[name]['best_params']
    if name == 'DecisionTree':
        return DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced', **bp)
    if name == 'RandomForest':
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced', **bp)
    if name == 'GradientBoosting':
        return GradientBoostingClassifier(random_state=RANDOM_STATE, **bp)
    if name == 'XGBoost':
        return xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_jobs=-1,
                                  scale_pos_weight=n_health_train / n_dep_train, **bp)
    if name == 'NaiveBayes':
        return GaussianNB(**bp)
    if name == 'LogisticRegression':
        return LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=2000, **bp)
    raise ValueError(f'Unknown model name: {name}')

# All base-model combinations tried in the 2-base-model stacking track
stacking_combinations = [
    (['DecisionTree', 'XGBoost'],         'LogisticRegression'),
    (['DecisionTree', 'GradientBoosting'],'LogisticRegression'),
    (['DecisionTree', 'RandomForest'],    'LogisticRegression'),
    (['DecisionTree', 'RandomForest'],    'XGBoost'),
    (['DecisionTree', 'XGBoost'],         'RandomForest'),
    (['DecisionTree', 'RandomForest'],    'GradientBoosting'),
    (['GradientBoosting', 'XGBoost'],     'LogisticRegression'),
    (['GradientBoosting', 'XGBoost'],     'RandomForest'),
    (['GradientBoosting', 'XGBoost'],     'DecisionTree'),
    (['NaiveBayes', 'DecisionTree'],      'LogisticRegression'),
    (['NaiveBayes', 'RandomForest'],      'LogisticRegression'),
    (['NaiveBayes', 'GradientBoosting'],  'LogisticRegression'),
    (['NaiveBayes', 'XGBoost'],           'LogisticRegression'),
]

METRIC_COLS = ['roc_auc', 'accuracy', 'recall', 'precision', 'specificity', 'f1']

print(f"{'='*60}\n  Stacking ensemble comparison\n{'='*60}")
stacking_summary = {}
for base_names, meta_name in stacking_combinations:
    estimators = [(n, make_estimator(n)) for n in base_names]
    final_est  = make_estimator(meta_name)
    stack = StackingClassifier(estimators=estimators, final_estimator=final_est,
                                cv=cv, stack_method='predict_proba', n_jobs=-1)
    stack.fit(X_train, y_train)

    m  = compute_metrics(stack, X_test, y_test)
    ci = bootstrap_ci(y_test, m['y_pred'], m['y_proba'])
    label = f"{'+'.join(base_names)}->{meta_name}"

    entry = {
        'best_params': {n: results[n]['best_params'] for n in base_names + [meta_name]},
        'cv_score': np.nan,
        'y_true': y_test, 'y_proba': m['y_proba'], 'ci': ci,
        **{k: v for k, v in m.items() if k not in ('y_pred', 'y_proba', 'tn', 'fp', 'fn', 'tp')},
    }
    results[f'Stacking({label})'] = entry
    stacking_summary[label] = entry

    def fmt(k): return f"{m[k]:.3f} [{ci[k][0]:.3f}–{ci[k][1]:.3f}]"
    print(f"\n  {label}")
    print(f"    ROC AUC      : {fmt('roc_auc')}")
    print(f"    Accuracy     : {fmt('accuracy')}")
    print(f"    Recall/Sens. : {fmt('recall')}")
    print(f"    Precision    : {fmt('precision')}")
    print(f"    Specificity  : {fmt('specificity')}")
    print(f"    F1           : {fmt('f1')}")

print('\n\n=== STACKING COMBINATIONS RANKED BY ROC-AUC (value [95% CI bootstrap]) ===')
hdr = (f"{'Combo':<35} {'ROC':>22} {'Accuracy':>22} {'Recall':>22} "
       f"{'Prec':>22} {'Spec':>22} {'F1':>22}")
print(hdr)
print('-' * len(hdr))
for label, entry in sorted(stacking_summary.items(), key=lambda x: x[1]['roc_auc'], reverse=True):
    ci = entry['ci']
    def f(k): return f"{entry[k]:.3f} [{ci[k][0]:.3f}–{ci[k][1]:.3f}]"
    print(f"{label:<35} {f('roc_auc'):>22} {f('accuracy'):>22} {f('recall'):>22} "
          f"{f('precision'):>22} {f('specificity'):>22} {f('f1'):>22}")

### 3-base-model stacking

Runs the 3-base-model track described above: every 3-model combination drawn from the 6-model
pool, combined with a `GradientBoosting` or `XGBoost` meta-model.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from itertools import combinations

def make_estimator(name):
    """Build a fresh estimator for `name` reusing its best_params from Step 4 GridSearchCV."""
    bp = results[name]['best_params']
    if name == 'DecisionTree':
        return DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced', **bp)
    if name == 'RandomForest':
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced', **bp)
    if name == 'GradientBoosting':
        return GradientBoostingClassifier(random_state=RANDOM_STATE, **bp)
    if name == 'XGBoost':
        return xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_jobs=-1,
                                  scale_pos_weight=n_health_train / n_dep_train, **bp)
    if name == 'NaiveBayes':
        return GaussianNB(**bp)
    if name == 'LogisticRegression':
        return LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=2000, **bp)
    raise ValueError(f'Unknown model name: {name}')

# All combinations of 3 base estimators drawn from this pool, tried against each meta-model
BASE_POOL     = ['DecisionTree', 'RandomForest', 'GradientBoosting', 'XGBoost', 'NaiveBayes', 'LogisticRegression']
META_NAMES    = ['GradientBoosting', 'XGBoost']
triple_combos = list(combinations(BASE_POOL, 3))

print(f"{'='*60}\n  Stacking ensemble comparison — 3 base models -> {META_NAMES}\n{'='*60}")
print(f'  {len(triple_combos)} base combinations x {len(META_NAMES)} meta-models '
      f'= {len(triple_combos) * len(META_NAMES)} total\n')

triple_summary = {}
for meta_name in META_NAMES:
    for base_names in triple_combos:
        base_names = list(base_names)
        estimators = [(n, make_estimator(n)) for n in base_names]
        final_est  = make_estimator(meta_name)
        stack = StackingClassifier(estimators=estimators, final_estimator=final_est,
                                    cv=cv, stack_method='predict_proba', n_jobs=-1)
        stack.fit(X_train, y_train)

        m  = compute_metrics(stack, X_test, y_test)
        ci = bootstrap_ci(y_test, m['y_pred'], m['y_proba'])
        label = f"{'+'.join(base_names)}->{meta_name}"

        entry = {
            'best_params': {n: results[n]['best_params'] for n in set(base_names + [meta_name])},
            'cv_score': np.nan,
            'y_true': y_test, 'y_proba': m['y_proba'], 'ci': ci,
            **{k: v for k, v in m.items() if k not in ('y_pred', 'y_proba', 'tn', 'fp', 'fn', 'tp')},
        }
        results[f'Stacking3({label})'] = entry
        triple_summary[label] = entry

print('=== 3-MODEL STACKING COMBINATIONS RANKED BY ROC-AUC (value [95% CI bootstrap]) ===')
hdr = (f"{'Combo':<55} {'ROC':>22} {'Accuracy':>22} {'Recall':>22} "
       f"{'Prec':>22} {'Spec':>22} {'F1':>22}")
print(hdr)
print('-' * len(hdr))
for label, entry in sorted(triple_summary.items(), key=lambda x: x[1]['roc_auc'], reverse=True):
    ci = entry['ci']
    def f(k): return f"{entry[k]:.3f} [{ci[k][0]:.3f}–{ci[k][1]:.3f}]"
    print(f"{label:<55} {f('roc_auc'):>22} {f('accuracy'):>22} {f('recall'):>22} "
          f"{f('precision'):>22} {f('specificity'):>22} {f('f1'):>22}")

## Step 6 — Youden's J threshold calibration

Steps 4-5 use the default 0.5 probability threshold. Here we **calibrate the decision
threshold** using **Youden's J statistic** (`J = TPR − FPR`, maximized), computed on
**out-of-fold TRAIN probabilities** (via `cross_val_predict`, same `cv` as Steps 4-5 — no
leakage) for the best-ranked model in each of three tracks:
- Best **single model** (Step 4, highest ROC-AUC)
- Best **2-base-model stacking** (Step 5)
- Best **3-base-model stacking** (Step 5)

The threshold found on TRAIN (out-of-fold) is then applied to the already-computed TEST
probabilities to recompute Accuracy / Precision / Recall / Specificity / F1 — ROC-AUC is
threshold-independent and stays the same. The three recalibrated tracks are ranked by F1 to
identify the most effective model overall after calibration.

Note: with very few TRAIN subjects, pooled out-of-fold probabilities can occasionally produce a
degenerate threshold (J≈0); this is reported as-is rather than silently corrected.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier


def make_estimator_full(name):
    """Like make_estimator, but also covers SVM and KNN (not used as stacking base/meta)."""
    bp = results[name]['best_params']
    if name == 'SVM':
        return SVC(random_state=RANDOM_STATE, probability=True, class_weight='balanced', **bp)
    if name == 'KNN':
        return KNeighborsClassifier(**bp)
    return make_estimator(name)


def youden_threshold(y_true, y_proba):
    """Threshold maximizing Youden's J = TPR - FPR."""
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    j = tpr - fpr
    best_idx = int(np.argmax(j))
    return float(thresholds[best_idx]), float(j[best_idx])


def recompute_with_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    ci = bootstrap_ci(y_true, y_pred, y_proba)
    return {
        'threshold': threshold, 'ci': ci,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def print_calibrated(label, y_true, y_proba, threshold, j_stat):
    m  = recompute_with_threshold(y_true, y_proba, threshold)
    ci = m['ci']
    def fmt(k): return f"{m[k]:.3f} [{ci[k][0]:.3f}–{ci[k][1]:.3f}]"
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    print(f"\n  Youden threshold : {threshold:.4f}  (J={j_stat:.4f})")
    print(f'\n  Confusion matrix:')
    print(f'                 Pred Health  Pred Dep')
    print(f"  True Health        {m['tn']:>3}         {m['fp']:>3}")
    print(f"  True Dep           {m['fn']:>3}         {m['tp']:>3}")
    print(f'\n  Test metrics  (value [95% CI bootstrap]):')
    print(f"    Accuracy     : {fmt('accuracy')}")
    print(f"    Precision    : {fmt('precision')}")
    print(f"    Recall/Sens. : {fmt('recall')}")
    print(f"    Specificity  : {fmt('specificity')}")
    print(f"    F1           : {fmt('f1')}")
    print(f"    ROC AUC      : {fmt('roc_auc')}  (unchanged by threshold)")
    return m


print(f"{'='*60}\n  Step 6 — Youden's J threshold calibration\n{'='*60}")

# ------------------------------------------------------------------
# (a) Best single model from Step 4
# ------------------------------------------------------------------
single_model_names = list(models_grids.keys())
best_single_name   = max(single_model_names, key=lambda n: results[n]['roc_auc'])
print(f"\nBest single model (Step 4, by ROC-AUC): {best_single_name}")

best_single_est  = make_estimator_full(best_single_name)
oof_proba_single = cross_val_predict(best_single_est, X_train, y_train, cv=cv,
                                      method='predict_proba')[:, 1]
thr_single, j_single = youden_threshold(y_train, oof_proba_single)
calib_single = print_calibrated(f'Single model: {best_single_name}',
                                 y_test, results[best_single_name]['y_proba'],
                                 thr_single, j_single)

# ------------------------------------------------------------------
# (b) Best 2-base-model stacking combination
# ------------------------------------------------------------------
best_2_label = max(stacking_summary.keys(), key=lambda l: stacking_summary[l]['roc_auc'])
print(f"\nBest 2-model stacking (by ROC-AUC): {best_2_label}")

base2_names, meta2_name = next(
    (b, m) for b, m in stacking_combinations if f"{'+'.join(b)}->{m}" == best_2_label
)
stack2 = StackingClassifier(
    estimators=[(n, make_estimator_full(n)) for n in base2_names],
    final_estimator=make_estimator_full(meta2_name),
    cv=cv, stack_method='predict_proba', n_jobs=-1,
)
oof_proba_2 = cross_val_predict(stack2, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
thr_2, j_2 = youden_threshold(y_train, oof_proba_2)
calib_2 = print_calibrated(f'2-model stacking: {best_2_label}',
                            y_test, stacking_summary[best_2_label]['y_proba'], thr_2, j_2)

# ------------------------------------------------------------------
# (c) Best 3-base-model stacking combination
# ------------------------------------------------------------------
best_3_label = max(triple_summary.keys(), key=lambda l: triple_summary[l]['roc_auc'])
print(f"\nBest 3-model stacking (by ROC-AUC): {best_3_label}")

base3_part, meta3_name = best_3_label.split('->')
base3_names = base3_part.split('+')
stack3 = StackingClassifier(
    estimators=[(n, make_estimator_full(n)) for n in base3_names],
    final_estimator=make_estimator_full(meta3_name),
    cv=cv, stack_method='predict_proba', n_jobs=-1,
)
oof_proba_3 = cross_val_predict(stack3, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
thr_3, j_3 = youden_threshold(y_train, oof_proba_3)
calib_3 = print_calibrated(f'3-model stacking: {best_3_label}',
                            y_test, triple_summary[best_3_label]['y_proba'], thr_3, j_3)

# ------------------------------------------------------------------
# Summary across the three calibrated tracks
# ------------------------------------------------------------------
calibrated_tracks = {
    f'Single: {best_single_name}': calib_single,
    f'Stacking2: {best_2_label}': calib_2,
    f'Stacking3: {best_3_label}': calib_3,
}
print('\n\n=== BEST MODEL PER TRACK AFTER YOUDEN CALIBRATION  (value [95% CI bootstrap]) ===')
hdr = (f"{'Track':<45} {'ROC':>22} {'Accuracy':>22} {'Recall':>22} "
       f"{'Prec':>22} {'Spec':>22} {'F1':>22}")
print(hdr)
print('-' * len(hdr))
for label, m in sorted(calibrated_tracks.items(), key=lambda x: x[1]['f1'], reverse=True):
    ci = m['ci']
    def f(k): return f"{m[k]:.3f} [{ci[k][0]:.3f}–{ci[k][1]:.3f}]"
    print(f"{label:<45} {f('roc_auc'):>22} {f('accuracy'):>22} {f('recall'):>22} "
          f"{f('precision'):>22} {f('specificity'):>22} {f('f1'):>22}")

overall_best_label = max(calibrated_tracks, key=lambda l: calibrated_tracks[l]['f1'])
print(f"\nMost effective model overall after threshold calibration: {overall_best_label}")

## Step 6D — Leakage-free comparison of all 8 individual models

For each of the 8 base models, computes out-of-fold TRAIN-CV metrics (via `cross_val_predict`)
at the default 0.5 threshold — a fully leakage-free comparison across all 8, independent of
Step 6's threshold-calibrated winner. The TEST AUC-ROC column is kept only as a reference,
never used to rank or select.

In [ ]:
# =====================================================================
# Step 6D -- Fully leakage-free comparison of the 8 individual models
# (out-of-fold TRAIN-CV metrics at default threshold 0.5, all 8; single
# TEST AUC-ROC per model kept only as a reference column, never used to
# rank or select)
# =====================================================================
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

def oof_metrics_at_threshold(estimator, X_tr, y_tr, cv, thr=0.5):
    oof_proba = cross_val_predict(estimator, X_tr, y_tr, cv=cv,
                                   method='predict_proba', n_jobs=-1)[:, 1]
    oof_pred = (oof_proba >= thr).astype(int)
    tn = ((oof_pred == 0) & (y_tr == 0)).sum()
    fp = ((oof_pred == 1) & (y_tr == 0)).sum()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    return {
        'roc_auc': roc_auc_score(y_tr, oof_proba),
        'accuracy': accuracy_score(y_tr, oof_pred),
        'precision': precision_score(y_tr, oof_pred, zero_division=0),
        'recall': recall_score(y_tr, oof_pred, zero_division=0),
        'specificity': specificity,
        'f1': f1_score(y_tr, oof_pred, zero_division=0),
    }

print(f"{'Model':<20} {'AUC(CV)':>8} {'Acc(CV)':>8} {'Sens(CV)':>9} {'Prec(CV)':>9} {'Spec(CV)':>9} {'F1(CV)':>8} {'AUC(TEST,ref)':>14}")
print('-' * 100)
for name in single_model_names:
    m = oof_metrics_at_threshold(make_estimator_full(name), X_train, y_train, cv, thr=0.5)
    test_auc = results[name]['roc_auc']
    print(f"{name:<20} {m['roc_auc']:>8.3f} {m['accuracy']:>8.3f} {m['recall']:>9.3f} "
          f"{m['precision']:>9.3f} {m['specificity']:>9.3f} {m['f1']:>8.3f} {test_auc:>14.3f}")

### Export out-of-fold probabilities

Saves each model's TRAIN labels and out-of-fold predicted probabilities to a JSON file
(`oof_export_<sex>_<window>.json`), used later for the nested/TRAIN-CV comparison figures.

In [ ]:
import json
from sklearn.model_selection import cross_val_predict

oof_export = {'y_train': [int(v) for v in y_train]}
for name in single_model_names:
    oof_proba = cross_val_predict(make_estimator_full(name), X_train, y_train, cv=cv,
                                   method='predict_proba', n_jobs=-1)[:, 1]
    oof_export[name] = [float(v) for v in oof_proba]

oof_export_path = f'oof_export_{SEX.lower()}_{WINDOW_LABEL}.json'
with open(oof_export_path, 'w') as f:
    json.dump(oof_export, f, indent=2)

print(f'Saved {oof_export_path} with y_train + out-of-fold probabilities for',
      len(single_model_names), 'models')

## Step 6B — Leakage-free model selection (TRAIN-CV, not TEST)

Re-selects the winning single model, 2-model stack, and 3-model stack using only out-of-fold
TRAIN-CV AUC-ROC — TEST is never consulted during selection, unlike Step 6 (which selects by
TEST AUC-ROC among all 61 candidates). Only after a winner is fixed per track does the cell
look at its TEST performance, a single look rather than a comparison across candidates. See
the cell's own header comment for the full rationale.

In [ ]:
# =====================================================================
# Step 6B -- Leakage-free model selection (choose the winner by TRAIN-CV
# AUC-ROC, not by TEST AUC-ROC)
# =====================================================================
# WHY THIS CELL EXISTS
# ---------------------
# Step 6 above selects the "best" individual model, best 2-model stacking,
# and best 3-model stacking by ranking all 61 candidates on their TEST-set
# ROC-AUC (`results[n]['roc_auc']`, `stacking_summary[...]['roc_auc']`,
# `triple_summary[...]['roc_auc']`, all computed via `compute_metrics(...,
# X_test, y_test)` back in Steps 4/5). Whoever happens to score highest on
# this ONE TEST set is crowned "the winner" -- so TEST is not a single,
# untouched held-out evaluation anymore, it is itself a model-selection
# resource, and the reported TEST metrics for "the winner" are
# optimistically biased by construction. The winner's-curse experiments
# (Step 7) quantify this bias after the fact; they do not undo it.
#
# This cell re-selects each of the three winners using ONLY TRAIN data:
# out-of-fold cross-validated probabilities (`cross_val_predict`, same
# `cv` already used throughout, so every subject's OOF probability comes
# from a fold that never saw that subject during fitting) -> ROC-AUC on
# those OOF probabilities. TEST is not consulted anywhere during this
# selection. Only once a winner is fixed per track do we look at its
# (already-computed, in Steps 4/5) TEST performance -- a single look, not
# a comparison across 61 candidates.
#
# PLACEMENT: run this AFTER Step 6 (it reuses `results`, `stacking_summary`,
# `triple_summary`, `best_single_name`, `best_2_label`, `best_3_label`,
# `overall_best_label`, `make_estimator_full`, `youden_threshold`,
# `print_calibrated`, all already defined there).
#
# COST: re-fits each of the 61 already-tuned candidates via 5-fold CV, so
# roughly 5x the compute of Steps 4/5 for the individual models, and more
# for the 53 stacking combinations (each stacking fit already contains its
# own internal 5-fold CV for the OOF meta-features) -- expect this cell to
# take noticeably longer to run than Steps 4-5 combined.
# =====================================================================

from sklearn.model_selection import cross_val_predict

print(f"{'='*70}\nStep 6B -- TRAIN-CV model selection (leakage-free w.r.t. TEST)\n{'='*70}")


def train_cv_auc(estimator):
    """Out-of-fold TRAIN ROC-AUC via cross_val_predict -- TEST is never touched."""
    oof_proba = cross_val_predict(estimator, X_train, y_train, cv=cv,
                                   method='predict_proba', n_jobs=-1)[:, 1]
    return roc_auc_score(y_train, oof_proba)


# ------------------------------------------------------------------
# (a) 8 individual models
# ------------------------------------------------------------------
single_cv_auc = {name: train_cv_auc(make_estimator_full(name)) for name in single_model_names}
best_single_cv = max(single_cv_auc, key=single_cv_auc.get)

print('\nIndividual models -- TRAIN-CV AUC vs TEST AUC (informational only):')
for name in sorted(single_cv_auc, key=single_cv_auc.get, reverse=True):
    flag = '  <-- TRAIN-CV winner' if name == best_single_cv else ''
    flag += '  <-- TEST winner (old Step 6)' if name == best_single_name else ''
    print(f"  {name:<20} TRAIN-CV AUC={single_cv_auc[name]:.3f}   "
          f"TEST AUC={results[name]['roc_auc']:.3f}{flag}")

# ------------------------------------------------------------------
# (b) 13 2-model stacking combinations
# ------------------------------------------------------------------
stack2_cv_auc = {}
for base_names, meta_name in stacking_combinations:
    label = f"{'+'.join(base_names)}->{meta_name}"
    stack = StackingClassifier(
        estimators=[(n, make_estimator_full(n)) for n in base_names],
        final_estimator=make_estimator_full(meta_name),
        cv=cv, stack_method='predict_proba', n_jobs=-1,
    )
    stack2_cv_auc[label] = train_cv_auc(stack)

best_2_cv = max(stack2_cv_auc, key=stack2_cv_auc.get)

print('\n2-model stacking -- TRAIN-CV AUC vs TEST AUC (informational only):')
for label in sorted(stack2_cv_auc, key=stack2_cv_auc.get, reverse=True):
    flag = '  <-- TRAIN-CV winner' if label == best_2_cv else ''
    flag += '  <-- TEST winner (old Step 6)' if label == best_2_label else ''
    print(f"  {label:<35} TRAIN-CV AUC={stack2_cv_auc[label]:.3f}   "
          f"TEST AUC={stacking_summary[label]['roc_auc']:.3f}{flag}")

# ------------------------------------------------------------------
# (c) 40 3-model stacking combinations
# ------------------------------------------------------------------
stack3_cv_auc = {}
for label in triple_summary:
    base3_part, meta3_name = label.split('->')
    base3_names = base3_part.split('+')
    stack = StackingClassifier(
        estimators=[(n, make_estimator_full(n)) for n in base3_names],
        final_estimator=make_estimator_full(meta3_name),
        cv=cv, stack_method='predict_proba', n_jobs=-1,
    )
    stack3_cv_auc[label] = train_cv_auc(stack)

best_3_cv = max(stack3_cv_auc, key=stack3_cv_auc.get)

print('\n3-model stacking -- TRAIN-CV AUC vs TEST AUC (top 10 by TRAIN-CV, informational):')
for label in sorted(stack3_cv_auc, key=stack3_cv_auc.get, reverse=True)[:10]:
    flag = '  <-- TRAIN-CV winner' if label == best_3_cv else ''
    flag += '  <-- TEST winner (old Step 6)' if label == best_3_label else ''
    print(f"  {label:<55} TRAIN-CV AUC={stack3_cv_auc[label]:.3f}   "
          f"TEST AUC={triple_summary[label]['roc_auc']:.3f}{flag}")

print(f"\n{'='*70}\nAgreement check: does TRAIN-CV selection match the original TEST-based\n"
      f"selection (Step 6)?\n{'='*70}")
print(f"  Individual : TRAIN-CV={best_single_cv!r:<40} TEST={best_single_name!r}"
      f"  {'MATCH' if best_single_cv == best_single_name else 'DIFFERENT'}")
print(f"  Stacking-2 : TRAIN-CV={best_2_cv!r:<40} TEST={best_2_label!r}"
      f"  {'MATCH' if best_2_cv == best_2_label else 'DIFFERENT'}")
print(f"  Stacking-3 : TRAIN-CV={best_3_cv!r:<40} TEST={best_3_label!r}"
      f"  {'MATCH' if best_3_cv == best_3_label else 'DIFFERENT'}")

# ------------------------------------------------------------------
# Youden calibration + single-look TEST evaluation of the TRAIN-CV
# winners (mirrors Step 6 exactly, just applied to the new winners)
# ------------------------------------------------------------------
print(f"\n{'='*70}\nYouden calibration on TRAIN-CV-selected winners "
      f"(single look at TEST, no TEST-based selection)\n{'='*70}")

# (a) single
best_single_cv_est  = make_estimator_full(best_single_cv)
oof_proba_single_cv = cross_val_predict(best_single_cv_est, X_train, y_train, cv=cv,
                                         method='predict_proba')[:, 1]
thr_single_cv, j_single_cv = youden_threshold(y_train, oof_proba_single_cv)
calib_single_cv = print_calibrated(f'[TRAIN-CV] Single model: {best_single_cv}',
                                    y_test, results[best_single_cv]['y_proba'],
                                    thr_single_cv, j_single_cv)

# (b) 2-model stacking
base2n, meta2n = next((b, m) for b, m in stacking_combinations
                       if f"{'+'.join(b)}->{m}" == best_2_cv)
stack2_cv = StackingClassifier(
    estimators=[(n, make_estimator_full(n)) for n in base2n],
    final_estimator=make_estimator_full(meta2n),
    cv=cv, stack_method='predict_proba', n_jobs=-1,
)
oof_proba_2_cv = cross_val_predict(stack2_cv, X_train, y_train, cv=cv,
                                    method='predict_proba')[:, 1]
thr_2_cv, j_2_cv = youden_threshold(y_train, oof_proba_2_cv)
calib_2_cv = print_calibrated(f'[TRAIN-CV] 2-model stacking: {best_2_cv}',
                               y_test, stacking_summary[best_2_cv]['y_proba'],
                               thr_2_cv, j_2_cv)

# (c) 3-model stacking
base3part_cv, meta3n_cv = best_3_cv.split('->')
base3n_cv = base3part_cv.split('+')
stack3_cv = StackingClassifier(
    estimators=[(n, make_estimator_full(n)) for n in base3n_cv],
    final_estimator=make_estimator_full(meta3n_cv),
    cv=cv, stack_method='predict_proba', n_jobs=-1,
)
oof_proba_3_cv = cross_val_predict(stack3_cv, X_train, y_train, cv=cv,
                                    method='predict_proba')[:, 1]
thr_3_cv, j_3_cv = youden_threshold(y_train, oof_proba_3_cv)
calib_3_cv = print_calibrated(f'[TRAIN-CV] 3-model stacking: {best_3_cv}',
                               y_test, triple_summary[best_3_cv]['y_proba'],
                               thr_3_cv, j_3_cv)

calibrated_tracks_cv = {
    f'Single: {best_single_cv}':   calib_single_cv,
    f'Stacking2: {best_2_cv}':     calib_2_cv,
    f'Stacking3: {best_3_cv}':     calib_3_cv,
}
print('\n\n=== [TRAIN-CV SELECTION] BEST MODEL PER TRACK AFTER YOUDEN CALIBRATION '
      '(value [95% CI bootstrap]) ===')
hdr = (f"{'Track':<45} {'ROC':>22} {'Accuracy':>22} {'Recall':>22} "
       f"{'Prec':>22} {'Spec':>22} {'F1':>22}")
print(hdr)
print('-' * len(hdr))
for label, m in sorted(calibrated_tracks_cv.items(), key=lambda x: x[1]['f1'], reverse=True):
    ci = m['ci']
    def f(k): return f"{m[k]:.3f} [{ci[k][0]:.3f}–{ci[k][1]:.3f}]"
    print(f"{label:<45} {f('roc_auc'):>22} {f('accuracy'):>22} {f('recall'):>22} "
          f"{f('precision'):>22} {f('specificity'):>22} {f('f1'):>22}")

overall_best_cv = max(calibrated_tracks_cv, key=lambda l: calibrated_tracks_cv[l]['f1'])
print(f"\nMost effective model overall after TRAIN-CV selection + calibration: "
      f"{overall_best_cv}")

print(f"\n{'='*70}\nFor comparison, the paper's current (TEST-selected) headline result is "
      f"the 'overall_best_label' from Step 6 above ({overall_best_label!r}).\n"
      f"Compare its F1/AUC to the TRAIN-CV-selected {overall_best_cv!r} above: if they "
      f"differ substantially, the gap is (at least in part) an estimate of how much the "
      f"original TEST-based selection was overfitting to this specific TEST set. If they "
      f"match or are close, that is evidence the original TEST-based winner was not "
      f"purely an artifact of TEST-set noise.\n{'='*70}")

## Step 6C — Redirect Step 7 to the TRAIN-CV-selected winner

Swaps `overall_best_label` for `overall_best_cv` (the Step 6B winner), so the winner's-curse
experiments in Step 7 evaluate the leakage-free winner instead of the original TEST-selected
one.

In [ ]:
# =====================================================================
# Step 6C -- Redirect Step 7 (Winner's Curse) to the TRAIN-CV-selected
# winner instead of the TEST-selected one
# =====================================================================
# WHY THIS CELL EXISTS
# ---------------------
# Step 7 (winner's curse) fixes ONE model -- via `build_fixed_model()`,
# which reads the `overall_best_label` string -- and runs Experiments
# 1/2/3a/3b on it. As written, `overall_best_label` still comes from the
# original Step 6 (TEST-based selection among the 61 candidates).
#
# This one-line cell swaps it for `overall_best_cv`, the winner chosen in
# Step 6B using only TRAIN-CV performance (TEST never consulted during
# selection). The string format is identical ('Single: ...',
# 'Stacking2: ...', 'Stacking3: ...'), so `build_fixed_model()` in Step 7
# needs no other change.
#
# PLACEMENT: paste this cell AFTER Step 6B and BEFORE Step 7, then run
# Step 7 as usual. The old Step 7 results (TEST-selected winner) you
# already have from before do NOT need to be re-run -- keep those outputs,
# this is an additional run on the new winner, not a replacement of them.
#
# COST: same as the original Step 7 run (~30-90 min per notebook).
# =====================================================================

overall_best_label = overall_best_cv
print(f"Step 7 (Winner's Curse) will now evaluate the TRAIN-CV-selected "
      f"winner: {overall_best_label}")
print("(Old Step 6 TEST-selected winner results you already have are "
      "untouched -- keep them for comparison.)")

## Step 7 — Winner's-curse / model-selection-bias check

Quantifies how much of the fixed winner's TEST performance is inflated by model-selection
bias: Experiments 1-2 repeat the *full* 61-candidate selection under varied seeds/splits (an
inflated, TEST-informed reference), while Experiments 3a-3b repeat only the *already-fixed*
winner under varied seeds/splits (the paper's reported, honest estimate). See the cell's own
header comment for the seed-propagation bug fix applied here.

In [ ]:
# =====================================================================
# Step 7 (FIXED) -- Winner's curse / model selection bias check
# =====================================================================
# Same as the original Step 7 cell, with one bug fixed: `_fit_auc` now
# propagates `seed` correctly even when the fixed/candidate model is a
# StackingClassifier. StackingClassifier has NO top-level `random_state`
# parameter (only its base estimators and final_estimator do), so the
# original `hasattr(est, 'random_state')` check silently failed for any
# stacking model, meaning Exp 1/3a fit the exact same deterministic
# stacking model across all "varied seed" repetitions instead of really
# varying it. This is invisible whenever the winner is a single model
# (its own random_state IS picked up correctly), but produces a
# degenerate SD=0.000 / point CI whenever the winner (or an Exp-1
# candidate) is a stacking combination.
#
# WARNING: 61 candidates x N_REPS repetitions is computationally heavy.
#   Each stacking fit runs its own internal CV. Expect 30-90 min for N_REPS=10.
#   Reduce N_REPS to 5 if computation time is a concern.
# =====================================================================

import math
import time
import warnings
from itertools import combinations as _combinations
from sklearn.base import clone
from sklearn.ensemble import StackingClassifier

warnings.filterwarnings('ignore')

N_REPS = 10


# -- Seed propagation, fixed for stacking models --------------------------------
def _set_seed(est, seed):
    """Set random_state on est itself, and (for StackingClassifier, which has no top-level random_state) on each of its base estimators and its final_estimator, wherever that parameter exists."""
    if hasattr(est, 'random_state'):
        est.set_params(random_state=seed)
    if isinstance(est, StackingClassifier):
        for _, sub_est in est.estimators:
            if hasattr(sub_est, 'random_state'):
                sub_est.set_params(random_state=seed)
        if est.final_estimator is not None and hasattr(est.final_estimator, 'random_state'):
            est.final_estimator.set_params(random_state=seed)
    return est


# -- Build stacking model from label string (reuses make_estimator_full from Step 6) --
def _build_stacking(combo_str):
    base_part, meta_name = combo_str.split('->')
    base_names = base_part.split('+')
    return StackingClassifier(
        estimators=[(n, make_estimator_full(n)) for n in base_names],
        final_estimator=make_estimator_full(meta_name),
        cv=cv, stack_method='predict_proba', n_jobs=-1,
    )


def build_fixed_model():
    label = overall_best_label
    if label.startswith('Single: '):
        return make_estimator_full(label[len('Single: '):])
    if label.startswith('Stacking2: '):
        return _build_stacking(label[len('Stacking2: '):])
    if label.startswith('Stacking3: '):
        return _build_stacking(label[len('Stacking3: '):])
    raise ValueError(f'Cannot parse overall_best_label: {label!r}')


# -- Build the full candidate pool (61 models) -----------------------------------
_WC_BASE_POOL  = ['DecisionTree', 'RandomForest', 'GradientBoosting',
                  'XGBoost', 'NaiveBayes', 'LogisticRegression']
_WC_META_NAMES = ['GradientBoosting', 'XGBoost']


def build_all_candidates():
    cands = {}
    for name in models_grids:                               # 8 single models
        cands[f'S:{name}'] = make_estimator_full(name)
    for base_names, meta_name in stacking_combinations:     # 13 two-model stacks
        lbl = f"{'+'.join(base_names)}->{meta_name}"
        cands[f'2:{lbl}'] = _build_stacking(lbl)
    for meta_name in _WC_META_NAMES:                        # 40 three-model stacks
        for triple in _combinations(_WC_BASE_POOL, 3):
            lbl = f"{'+'.join(triple)}->{meta_name}"
            cands[f'3:{lbl}'] = _build_stacking(lbl)
    return cands


print(f"Fixed model (Youden winner): {overall_best_label}")
print(f"Candidate pool: {len(build_all_candidates())} models  (N_REPS={N_REPS})")
print()

# -- Collect all subject feature vectors -----------------------------------------
print("Building all-subject feature matrices from Step 3 outputs...")
t0 = time.time()

all_dep_X  = np.vstack([X_train[y_train == 1], X_test[y_test == 1]])
all_hlth_X = np.vstack([X_train[y_train == 0], X_test[y_test == 0]])

N_DEP        = all_dep_X.shape[0]
N_HLTH       = all_hlth_X.shape[0]
N_DEP_TEST   = int((y_test == 1).sum())
N_HLTH_TEST  = int((y_test == 0).sum())
N_DEP_TRAIN  = int((y_train == 1).sum())
N_HLTH_TRAIN = int((y_train == 0).sum())

print(f"  Done in {time.time()-t0:.1f}s. Subjects: {N_DEP} DEP, {N_HLTH} HEALTH")
print(f"  Test sizes preserved: {N_DEP_TEST} DEP, {N_HLTH_TEST} HEALTH")
print()


def _fit_auc(estimator, X_tr, y_tr, X_te, y_te, seed=None):
    """Clone, optionally set seed (recursively, incl. stacking sub-estimators), fit, return ROC-AUC (features already normalised)."""
    est = clone(estimator)
    if seed is not None:
        _set_seed(est, seed)
    est.fit(X_tr, y_tr)
    if len(np.unique(y_te)) < 2:
        return np.nan
    return roc_auc_score(y_te, est.predict_proba(X_te)[:, 1])


def _make_split(dep_perm, hlth_perm):
    """Build train/test matrices from subject-index permutations."""
    X_tr = np.vstack([all_dep_X[dep_perm[N_DEP_TEST:]],
                      all_hlth_X[hlth_perm[N_HLTH_TEST:]]])
    y_tr = np.concatenate([np.ones(N_DEP - N_DEP_TEST),
                           np.zeros(N_HLTH - N_HLTH_TEST)])
    X_te = np.vstack([all_dep_X[dep_perm[:N_DEP_TEST]],
                      all_hlth_X[hlth_perm[:N_HLTH_TEST]]])
    y_te = np.concatenate([np.ones(N_DEP_TEST), np.zeros(N_HLTH_TEST)])
    return X_tr, y_tr, X_te, y_te


# -- Exp 1: Adaptive, FIXED split, varied seeds -----------------------------------
print("Exp 1 — Adaptive (all 61 candidates), fixed split, varied seeds:")
exp1_aucs = []
for seed in range(N_REPS):
    t1 = time.time()
    best = 0.0
    for est in build_all_candidates().values():
        auc = _fit_auc(est, X_train, y_train, X_test, y_test, seed=seed)
        if not np.isnan(auc):
            best = max(best, auc)
    exp1_aucs.append(best)
    print(f"  seed={seed}  best_auc={best:.3f}  ({time.time()-t1:.0f}s)")

# -- Exp 2: Adaptive, VARIED splits -----------------------------------------------
print("\nExp 2 — Adaptive (all 61 candidates), varied splits:")
exp2_aucs = []
rng2 = np.random.default_rng(42)
for rep in range(N_REPS):
    t1 = time.time()
    dp = rng2.permutation(N_DEP)
    hp = rng2.permutation(N_HLTH)
    X_tr, y_tr, X_te, y_te = _make_split(dp, hp)
    if len(np.unique(y_te)) < 2:
        continue
    best = 0.0
    for est in build_all_candidates().values():
        auc = _fit_auc(est, X_tr, y_tr, X_te, y_te)
        if not np.isnan(auc):
            best = max(best, auc)
    exp2_aucs.append(best)
    print(f"  rep={rep}  best_auc={best:.3f}  ({time.time()-t1:.0f}s)")

# -- Exp 3a: Fixed model, FIXED split, varied seeds --------------------------------
print("\nExp 3a — Fixed model, fixed split, varied seeds:")
exp3a_aucs = []
for seed in range(N_REPS):
    auc = _fit_auc(build_fixed_model(), X_train, y_train, X_test, y_test, seed=seed)
    exp3a_aucs.append(auc)
    print(f"  seed={seed}  auc={auc:.3f}")

# -- Exp 3b: Fixed model, VARIED splits --------------------------------------------
print("\nExp 3b — Fixed model, varied splits:")
exp3b_aucs = []
rng3 = np.random.default_rng(99)
for rep in range(N_REPS):
    dp = rng3.permutation(N_DEP)
    hp = rng3.permutation(N_HLTH)
    X_tr, y_tr, X_te, y_te = _make_split(dp, hp)
    if len(np.unique(y_te)) < 2:
        continue
    auc = _fit_auc(build_fixed_model(), X_tr, y_tr, X_te, y_te)
    exp3b_aucs.append(auc)
    print(f"  rep={rep}  auc={auc:.3f}")

# -- Summary -------------------------------------------------------------------------
def _row(exp_label, sel_label, aucs):
    arr  = np.array([a for a in aucs if not np.isnan(a)])
    n    = len(arr)
    mean = arr.mean()
    std  = arr.std()
    se   = std / math.sqrt(n) if n > 1 else 0.0
    lo, hi = mean - 1.96 * se, mean + 1.96 * se
    return {
        'Experiment': exp_label,
        'Selection':  sel_label,
        'N': n, 'AUC mean': round(mean, 3), 'std': round(std, 3),
        'CI 95% lo': round(lo, 3), 'CI 95% hi': round(hi, 3),
        'Crosses 0.5': 'YES ***' if lo <= 0.5 <= hi else 'No',
    }

fixed_label = f'Fixed model: {overall_best_label}'
rows = [
    _row('1. Seeds, fixed split',  'Best per rep — 61 cands (inflated)', exp1_aucs),
    _row('2. Repeated splits',     'Best per rep — 61 cands (inflated)', exp2_aucs),
    _row('3a. Seeds, fixed split', fixed_label, exp3a_aucs),
    _row('3b. Repeated splits',    fixed_label, exp3b_aucs),
]

print()
print('=' * 110)
print("  WINNER'S CURSE / MODEL SELECTION BIAS CHECK")
print('=' * 110)
hdr = (f"{'Experiment':<25} {'Selection':<45} {'N':>3} "
       f"{'AUC mean':>9} {'std':>6} {'CI 95% approx':>18} {'Crosses 0.5':>12}")
print(hdr)
print('-' * len(hdr))
for r in rows:
    ic = f"[{r['CI 95% lo']:.3f}-{r['CI 95% hi']:.3f}]"
    print(f"{r['Experiment']:<25} {r['Selection']:<45} {r['N']:>3} "
          f"{r['AUC mean']:>9.3f} {r['std']:>6.3f} {ic:>18} {r['Crosses 0.5']:>12}")

print()
print("Interpretation:")
for r in rows:
    if 'Fixed model' in r['Selection']:
        ic = f"[{r['CI 95% lo']:.3f}-{r['CI 95% hi']:.3f}]"
        if r['Crosses 0.5'] == 'YES ***':
            print(f"  {r['Experiment']}: CI {ic} INCLUDES 0.5.")
            print(f"    Result is NOT distinguishable from chance. The model shows no real predictive capacity.")
        else:
            print(f"  {r['Experiment']}: CI {ic} does NOT cross 0.5. Evidence of genuine signal.")

pd.DataFrame(rows)[[
    'Experiment', 'Selection', 'N', 'AUC mean', 'std', 'CI 95% lo', 'CI 95% hi', 'Crosses 0.5'
]]